In [1]:
list.of.packages <- c("tidyverse","data.table","stargazer")
new.packages <- list.of.packages[!(list.of.packages %in% installed.packages()[,"Package"])]
if(length(new.packages)) install.packages(new.packages, repos = "http://cran.us.r-project.org")

invisible(lapply(list.of.packages, library, character.only = TRUE))

options(repr.matrix.max.rows=500, repr.matrix.max.cols=1200, message=FALSE, warning=TRUE)  

── Attaching core tidyverse packages ─────────────────────────────────────────────────────────────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.1     ✔ stringr   1.6.0
✔ ggplot2   4.0.1     ✔ tibble    3.3.0
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.2.0     
── Conflicts ───────────────────────────────────────────────────────────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::filter() masks stats::filter()
✖ dplyr::lag()    masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors

Attachement du package : ‘data.table’


Les objets suivants sont masqués depuis ‘package:lubridate’:

    hour, isoweek, mday, minute, month, quarter, second, wday, week,
    yday, year


Les objets suivants sont masqués depuis ‘package:dplyr’:

    between, first, last


L'objet suivant est masqué depuis ‘package:purrr’:

    transpose



Please cite as: 


 Hla

# Longer dataframe

In [2]:
df_deg_bins <- fread("./interm/df_WB_cells_CDC_longer.csv")
sprintf("%i x %i dataframe", nrow(df_deg_bins), ncol(df_deg_bins))
head(df_deg_bins,2)

[1] "30298932 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>
42474,91.75,27.50,T0DayMin1,1,0,1
42660,96.75,28.25,T0DayMin1,4,0,1


# Computation $$T_{i}^{30}= \sum_{b} \frac{h_{ib}^{30} \times T_b}{24 \times 30}$$

## 30-day exposure

In [3]:
result_30 <- df_deg_bins[, .(WB_i30 = sum(N_hours * Temp / 24 / 30, na.rm = TRUE)), 
                          by = .(Measured_date_CDC, cell_x, cell_y)]
sprintf("%i x %i dataframe", nrow(result_30), ncol(result_30))
head(result_30,2)

[1] "126725 x 4 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_i30
<int>,<dbl>,<dbl>,<dbl>
42474,91.75,27.50,5.580556
42660,96.75,28.25,10.943056


## 20-day exposure 

In [4]:
df_deg_bins_20 <- df_deg_bins[Day_Min <=20]
sprintf("%i x %i dataframe", nrow(df_deg_bins_20), ncol(df_deg_bins_20))
head(df_deg_bins,2)

[1] "20131561 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>
42474,91.75,27.50,T0DayMin1,1,0,1
42660,96.75,28.25,T0DayMin1,4,0,1


In [5]:
result_20 <- df_deg_bins_20[, .(WB_i20 = sum(N_hours * Temp / 24 / 20, na.rm = TRUE)), 
                          by = .(Measured_date_CDC, cell_x, cell_y)]
sprintf("%i x %i dataframe", nrow(result_20), ncol(result_20))
head(result_20,2)

[1] "126725 x 4 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_i20
<int>,<dbl>,<dbl>,<dbl>
42474,91.75,27.50,6.608333
42660,96.75,28.25,10.629167


## 10-day exposure

In [6]:
df_deg_bins_10 <- df_deg_bins[Day_Min <=10]
sprintf("%i x %i dataframe", nrow(df_deg_bins_10), ncol(df_deg_bins_10))
head(df_deg_bins_10,2)

[1] "10035276 x 7 dataframe"

Measured_date_CDC,cell_x,cell_y,Temp_bin,N_hours,Temp,Day_Min
<int>,<dbl>,<dbl>,<chr>,<int>,<int>,<int>
42474,91.75,27.50,T0DayMin1,1,0,1
42660,96.75,28.25,T0DayMin1,4,0,1


In [7]:
result_10 <- df_deg_bins_10[, .(WB_i10 = sum(N_hours * Temp / 24 / 10, na.rm = TRUE)), 
                          by = .(Measured_date_CDC, cell_x, cell_y)]
sprintf("%i x %i dataframe", nrow(result_10), ncol(result_10))
head(result_10,2)

[1] "126725 x 4 dataframe"

Measured_date_CDC,cell_x,cell_y,WB_i10
<int>,<dbl>,<dbl>,<dbl>
42474,91.75,27.50,7.283333
42660,96.75,28.25,9.354167


## MERGE

In [8]:
result <- data.frame(result_30) %>%
                left_join(result_20,by=c("Measured_date_CDC","cell_x","cell_y"))%>%
                left_join(result_10,by=c("Measured_date_CDC","cell_x","cell_y"))
sprintf("%i x %i dataframe", nrow(result), ncol(result))
head(result,2)

[1] "126725 x 6 dataframe"

,Measured_date_CDC,cell_x,cell_y,WB_i30,WB_i20,WB_i10
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>
1,42474,91.75,27.50,5.580556,6.608333,7.283333
2,42660,96.75,28.25,10.943056,10.629167,9.354167


In [9]:
data.frame(result) %>% 
        select(WB_i30,WB_i20,WB_i10)%>%
        stargazer(nobs=TRUE, digits = 2,
                  summary.stat = c("n","mean", "sd","min","p25","median","p75","max"),
                  title = c("Summary statistics"),
                  type="text"
                     )


Summary statistics
Statistic    N    Mean  St. Dev.  Min   Pctl(25) Median Pctl(75)  Max 
----------------------------------------------------------------------
WB_i30    126,725 20.69   6.08   -14.43  16.54   22.30   25.50   29.35
WB_i20    126,725 20.85   6.03   -14.99  16.87   22.52   25.57   29.62
WB_i10    126,725 21.03   5.98   -16.32  17.31   22.69   25.66   30.03
----------------------------------------------------------------------


# Categorize temperature into bins

### Wet-bulb temperature thresholds

In [10]:
thresholds_WB <- c(-17,14,16,18,20,22,24,26,28,31)

In [11]:
result_bins <- result %>%
                    mutate(WB_i30_bin = cut(WB_i30, breaks = thresholds_WB, 
                                           labels = c("<=14°C","(14,16]","(16,18]","(18,20]","(20,22]","(22,24]","(24,26]","(26,28]",">28°C"),
                                           include.lowest = TRUE),
                           WB_i20_bin = cut(WB_i20, breaks = thresholds_WB, 
                                           labels = c("<=14°C","(14,16]","(16,18]","(18,20]","(20,22]","(22,24]","(24,26]","(26,28]",">28°C"),
                                           include.lowest = TRUE),
                           WB_i10_bin = cut(WB_i10, breaks = thresholds_WB, 
                                           labels = c("<=14°C","(14,16]","(16,18]","(18,20]","(20,22]","(22,24]","(24,26]","(26,28]",">28°C"),
                                           include.lowest = TRUE)                      
                          )
sprintf("%i x %i dataframe", nrow(result_bins), ncol(result_bins))
head(result_bins,2)

[1] "126725 x 9 dataframe"

,Measured_date_CDC,cell_x,cell_y,WB_i30,WB_i20,WB_i10,WB_i30_bin,WB_i20_bin,WB_i10_bin
,<int>,<dbl>,<dbl>,<dbl>,<dbl>,<dbl>,<fct>,<fct>,<fct>
1,42474,91.75,27.50,5.580556,6.608333,7.283333,<=14°C,<=14°C,<=14°C
2,42660,96.75,28.25,10.943056,10.629167,9.354167,<=14°C,<=14°C,<=14°C


## SAVE

In [12]:
fwrite(result_bins,"./output/df_WB_cells_CDC_30_20_10.csv")